# RetinaFace evaluation across multiple Kaggle face-detection datasets

This notebook evaluates the **RetinaFace** detector on six Kaggle datasets and saves every metric, curve, confusion matrix and figure to `./data/` for later reference.

**Structure**
1. `FaceDataset` is a plain container that only stores preprocessed images and annotations.
2. Outside functions (`preprocess_face_sample`, `build_dataset`, `build_custom_dataset`) read files, preprocess images and create `FaceDataset` objects.
3. The `RetinaFace` class runs the detector: `evaluate(dataset)` computes metrics, saves CSV/JSON and plots figures; `compare(results)` builds the cross-dataset comparison and a combined PDF report.

**Before running:** turn on *Internet* in the Kaggle notebook settings (and a GPU accelerator for speed).

## Datasets (each slug verified to exist on Kaggle)

| Name | Kaggle dataset | Annotations | Notes |
|---|---|---|---|
| `wider_face` | [iamprateek/wider-face-a-face-detection-dataset](https://www.kaggle.com/datasets/iamprateek/wider-face-a-face-detection-dataset) | WIDER `bbx_gt.txt` | Validation split; `invalid` boxes skipped |
| `fddb` | [ngoduy/dataset-for-face-detection](https://www.kaggle.com/datasets/ngoduy/dataset-for-face-detection) | FDDB ellipse lists | Ellipses converted to bounding boxes |
| `celeba_aligned` | [jessicali9530/celeba-dataset](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset) | Aligned 5-point landmarks | Test partition; landmark-based matching (see below) |
| `face_mask_detection` | [andrewmvd/face-mask-detection](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection) | Pascal VOC XML | Masked / unmasked / incorrectly-masked faces |
| `open_images_faces` | [fareselmenshawii/face-detection-dataset](https://www.kaggle.com/datasets/fareselmenshawii/face-detection-dataset) | YOLO txt | ~16.7k Google Open Images photos |
| `human_faces` | [garidziracrispen/human-faces](https://www.kaggle.com/datasets/garidziracrispen/human-faces) | auto-detected | Loader tries VOC, COCO and YOLO formats |

Changes from the previous version: the old slugs `mksaad/wider-face-a-face-detection-benchmark` and `shashwatwork/fddb-dataset` could not be found on Kaggle and were replaced with the verified sources above.

**CelebA caveat:** `list_bbox_celeba.csv` describes the *original* in-the-wild photos, not the 178×218 aligned crops that this Kaggle dataset ships, so those boxes cannot be used. A detection on CelebA counts as correct when it contains all 5 aligned landmarks (and overlaps a landmark-derived approximate box by IoU ≥ 0.2). Because of this, IoU-based AP75 and mAP@.5:.95 are not reported for CelebA.

Every dataset is also **re-verified at runtime**: it must download and its annotations must parse, otherwise it is skipped and the reason is written to `data/dataset_verification.csv`.

## Metric definitions

| Metric | Definition |
|---|---|
| Precision, Recall, F1 | Detection level at `SCORE_THRESHOLD`, matching each detection greedily to an unmatched face with IoU ≥ `IOU_THRESHOLD` |
| Detection accuracy | TP / (TP + FP + FN). Detection has no countable true negatives, so the usual accuracy formula is undefined |
| AP50, AP75, mAP@.5:.95 | Area under the precision-recall curve (all-point interpolation) at IoU 0.50, 0.75 and averaged over 0.50 to 0.95 |
| Detection ROC-AUC | How well the confidence score separates correct detections from false detections |
| FROC | Recall versus false positives per image, the standard FDDB-style curve |
| Image accuracy, Image ROC-AUC | Image-level "contains a face?" decision using the highest detection score. ROC needs face-free images, so it is only defined when the sampled subset contains some |
| Count accuracy, Count MAE | Exact match and mean absolute error between predicted and true face counts per image |
| Recall by face size | Recall for tiny (<16 px), small (16–32), medium (32–96) and large (≥96) faces |

Detections are produced with `INFERENCE_THRESHOLD` (0.05) so that all curves span the whole confidence range; operating-point metrics use `SCORE_THRESHOLD` (0.90).

## Setup

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# Enable "Internet" in the Kaggle notebook settings before running.
# ============================================================
%pip -q install retina-face kagglehub pandas numpy pillow tqdm matplotlib scikit-learn

# retina-face runs on TensorFlow. TF >= 2.16 ships Keras 3, so retina-face
# needs the matching tf-keras package. Pin it to the installed TF version
# so pip does not silently upgrade TensorFlow itself.
import importlib.util, subprocess, sys
if importlib.util.find_spec("tensorflow") is not None:
    import tensorflow as tf
    major, minor = (int(p) for p in tf.__version__.split(".")[:2])
    if (major, minor) >= (2, 16):
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"tf-keras~={major}.{minor}.0"],
            check=False,
        )


In [ ]:
# ============================================================
# IMPORTS & HARDWARE CHECK
# ============================================================
import gc
import io
import json
import math
import time
import zipfile
import warnings
import functools
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

# The library class is imported under an alias so that our own evaluation
# class can be named `RetinaFace`.
from retinaface import RetinaFace as RetinaFaceLib
import kagglehub

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(gpus), [g.name for g in gpus])

plt.rcParams.update({
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
OUTPUT_DIR = Path("data")                 # every metric, table and figure is saved here
DOWNLOAD_DIR = Path("kaggle_datasets")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
DOWNLOAD_DIR.mkdir(exist_ok=True, parents=True)

MAX_IMAGES_PER_DATASET = 500   # None = evaluate every annotated image (slow on WIDER/CelebA)
RANDOM_SEED = 42               # images are randomly sampled (not "first N") so subsets are representative

# Operating point: precision / recall / F1 / accuracy / confusion matrices use this score.
SCORE_THRESHOLD = 0.90
# RetinaFace is run with a much lower threshold so the ROC, PR and threshold-sweep
# curves cover the full confidence range instead of being truncated at 0.90.
INFERENCE_THRESHOLD = 0.05
IOU_THRESHOLD = 0.50

# Preprocessing: images whose longest side exceeds this are downscaled
# (boxes and landmarks are rescaled with them). None keeps original resolution.
MAX_IMAGE_SIDE = 1600

FIGURE_DPI = 150
NUM_VISUAL_SAMPLES = 8          # images drawn in each dataset's prediction grid

# Every slug below was checked to exist on Kaggle. The notebook re-verifies them at
# runtime (download + annotation parse) and only evaluates datasets that pass.
DATASETS = [
    {
        "name": "wider_face",
        "kaggle_handle": "iamprateek/wider-face-a-face-detection-dataset",
        "builder": "wider",
        "description": "WIDER FACE validation split: crowds, tiny faces, heavy occlusion.",
    },
    {
        "name": "fddb",
        "kaggle_handle": "ngoduy/dataset-for-face-detection",
        "builder": "fddb",
        "description": "FDDB (2,845 images / 5,171 faces, ellipse annotations -> boxes).",
    },
    {
        "name": "celeba_aligned",
        "kaggle_handle": "jessicali9530/celeba-dataset",
        "builder": "celeba",
        "description": "CelebA aligned crops, one face each; matched via the 5 aligned landmarks.",
    },
    {
        "name": "face_mask_detection",
        "kaggle_handle": "andrewmvd/face-mask-detection",
        "builder": "voc",
        "description": "853 images, Pascal VOC boxes, masked / unmasked / incorrect-mask faces.",
    },
    {
        "name": "open_images_faces",
        "kaggle_handle": "fareselmenshawii/face-detection-dataset",
        "builder": "yolo",
        "description": "~16.7k Google Open Images photos with YOLO-format face boxes.",
    },
    {
        "name": "human_faces",
        "kaggle_handle": "garidziracrispen/human-faces",
        "builder": "auto",
        "description": "Human face detection set; annotation format auto-detected (VOC/COCO/YOLO).",
    },
]


## Dataset container

In [ ]:
# ============================================================
# DATASET CONTAINER
# `FaceDataset` only STORES data. It never reads files or transforms
# images; that is done by the outside preprocessing functions below.
# ============================================================

class FaceDataset:
    """
    Holds already-preprocessed images and their annotations.

    images     : list of RGB uint8 arrays (H, W, 3)
    boxes      : list of float32 arrays (N, 4), pixel xyxy, one per image
    image_ids  : list of str
    landmarks  : optional list of float32 arrays (N, 5, 2) aligned with `boxes`
    match_mode : "iou"       -> a detection is correct if IoU >= threshold
                 "landmarks" -> a detection is correct if it contains all 5 landmarks
                               (used when the GT boxes do not fit the images, e.g. CelebA aligned)
    metadata   : free-form dict describing the source and preprocessing
    """

    def __init__(self, name, images, boxes, image_ids,
                 landmarks=None, match_mode="iou", metadata=None):
        if not (len(images) == len(boxes) == len(image_ids)):
            raise ValueError("images, boxes and image_ids must have the same length")
        if landmarks is not None and len(landmarks) != len(images):
            raise ValueError("landmarks must have one entry per image")
        if match_mode not in {"iou", "landmarks"}:
            raise ValueError("match_mode must be 'iou' or 'landmarks'")
        if match_mode == "landmarks" and landmarks is None:
            raise ValueError("match_mode='landmarks' requires landmarks")

        self.name = name
        self.images = list(images)
        self.boxes = [np.asarray(b, dtype=np.float32).reshape(-1, 4) for b in boxes]
        self.image_ids = list(image_ids)
        self.landmarks = (
            None if landmarks is None
            else [np.asarray(l, dtype=np.float32).reshape(-1, 5, 2) for l in landmarks]
        )
        self.match_mode = match_mode
        self.metadata = dict(metadata or {})

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return {
            "image": self.images[index],
            "boxes": self.boxes[index],
            "landmarks": None if self.landmarks is None else self.landmarks[index],
            "image_id": self.image_ids[index],
        }

    def __iter__(self):
        for index in range(len(self)):
            yield self[index]

    @property
    def num_faces(self):
        return int(sum(len(b) for b in self.boxes))

    def summary(self):
        faces = np.array([len(b) for b in self.boxes]) if self.boxes else np.zeros(0)
        heights = np.array([img.shape[0] for img in self.images]) if self.images else np.zeros(0)
        widths = np.array([img.shape[1] for img in self.images]) if self.images else np.zeros(0)
        sizes = np.concatenate([
            np.sqrt(np.clip((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]), 0, None))
            for b in self.boxes if len(b)
        ]) if self.num_faces else np.zeros(0)
        return {
            "dataset": self.name,
            "images": len(self),
            "faces": self.num_faces,
            "images_without_faces": int((faces == 0).sum()),
            "mean_faces_per_image": float(faces.mean()) if len(faces) else 0.0,
            "max_faces_per_image": int(faces.max()) if len(faces) else 0,
            "mean_width": float(widths.mean()) if len(widths) else 0.0,
            "mean_height": float(heights.mean()) if len(heights) else 0.0,
            "median_face_size_px": float(np.median(sizes)) if len(sizes) else float("nan"),
            "match_mode": self.match_mode,
        }

    def __repr__(self):
        return f"FaceDataset(name={self.name!r}, images={len(self)}, faces={self.num_faces}, match_mode={self.match_mode!r})"


## Preprocessing and dataset builders (outside the class)

In [ ]:
# ============================================================
# CUSTOM PREPROCESSING (outside the dataset class)
# Swap `preprocess_face_sample` for your own function with the same
# signature to change how images are prepared before evaluation.
# ============================================================

def preprocess_face_sample(image, boxes, landmarks=None, max_side=MAX_IMAGE_SIDE):
    """
    image     : PIL.Image
    boxes     : (N, 4) pixel xyxy in the ORIGINAL image
    landmarks : optional (N, 5, 2)
    returns   : (RGB uint8 array, clipped/rescaled boxes, rescaled landmarks)
    """
    image = image.convert("RGB")
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    if landmarks is not None:
        landmarks = np.asarray(landmarks, dtype=np.float32).reshape(-1, 5, 2)

    scale = 1.0
    if max_side is not None and max(image.size) > max_side:
        scale = max_side / float(max(image.size))
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.BILINEAR)

    width, height = image.size
    boxes = boxes * scale
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, width)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, height)
    keep = (boxes[:, 2] - boxes[:, 0] > 1) & (boxes[:, 3] - boxes[:, 1] > 1)
    boxes = boxes[keep]
    if landmarks is not None:
        landmarks = landmarks[keep] * scale

    return np.asarray(image, dtype=np.uint8), boxes, landmarks


def make_face_dataset(name, raw_samples, preprocess_fn=preprocess_face_sample,
                      match_mode="iou", limit=None, seed=RANDOM_SEED, metadata=None):
    """
    raw_samples: list of dicts {"load": callable -> PIL.Image, "boxes": (N,4),
                               "landmarks": optional (N,5,2), "image_id": str}
    Samples `limit` items, runs `preprocess_fn` on each, returns a FaceDataset.
    """
    if not raw_samples:
        raise ValueError(f"{name}: no annotated images were found")

    rng = np.random.default_rng(seed)
    if limit is not None and len(raw_samples) > limit:
        chosen = sorted(rng.choice(len(raw_samples), size=limit, replace=False))
        raw_samples = [raw_samples[i] for i in chosen]

    images, boxes, landmarks, ids = [], [], [], []
    use_landmarks = match_mode == "landmarks"

    for sample in tqdm(raw_samples, desc=f"Preprocessing {name}"):
        try:
            image = sample["load"]()
            array, b, l = preprocess_fn(image, sample["boxes"], sample.get("landmarks"))
        except Exception as exc:
            warnings.warn(f"{name}: skipping {sample['image_id']}: {exc}")
            continue
        images.append(array)
        boxes.append(b)
        landmarks.append(l if l is not None else np.zeros((0, 5, 2), np.float32))
        ids.append(sample["image_id"])

    meta = {"preprocess_fn": getattr(preprocess_fn, "__name__", str(preprocess_fn)),
            "max_image_side": MAX_IMAGE_SIDE, "available_annotated_images": None}
    meta.update(metadata or {})
    return FaceDataset(name, images, boxes, ids,
                       landmarks=landmarks if use_landmarks else None,
                       match_mode=match_mode, metadata=meta)


# ------------------------------------------------------------
# File helpers
# ------------------------------------------------------------
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(root):
    return [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]


def path_key(path, depth):
    """Lower-case key built from the last `depth` path parts, extension removed."""
    parts = list(Path(path).with_suffix("").parts[-depth:])
    return "/".join(parts).lower()


def file_loader(path):
    return lambda: Image.open(path)


@functools.lru_cache(maxsize=4)
def _open_zip(path):
    return zipfile.ZipFile(path)


def zip_loader(zip_path, member):
    return lambda: Image.open(io.BytesIO(_open_zip(str(zip_path)).read(member)))


def prefer_split(paths, preferred=("val", "valid", "validation", "test")):
    """Sort so files from evaluation splits come first (train last)."""
    def rank(p):
        text = str(p).lower()
        return 0 if any(f"/{s}" in text or f"_{s}" in text for s in preferred) else 1
    return sorted(paths, key=lambda p: (rank(p), str(p)))


# ------------------------------------------------------------
# WIDER FACE  (wider_face_{val,train}_bbx_gt.txt)
# ------------------------------------------------------------
def collect_wider_face(root):
    root = Path(root)
    gt_files = [p for p in root.rglob("*.txt")
                if "wider_face" in p.name.lower() and "bbx_gt" in p.name.lower()]
    if not gt_files:
        raise FileNotFoundError("WIDER FACE *_bbx_gt.txt annotation file not found")
    gt_file = prefer_split(gt_files)[0]          # validation split preferred
    split = "val" if "val" in gt_file.name.lower() else "train"

    image_index = {}
    for p in list_images(root):
        image_index.setdefault(path_key(p, 2), []).append(p)

    lines = gt_file.read_text(encoding="utf-8", errors="replace").splitlines()
    samples, i = [], 0
    while i < len(lines):
        rel = lines[i].strip(); i += 1
        if not rel.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        try:
            count = int(lines[i].strip()); i += 1
        except (ValueError, IndexError):
            continue
        # Images with 0 faces still have one placeholder line "0 0 0 0 0 0 0 0 0 0".
        box_lines = lines[i:i + max(count, 1)]
        i += max(count, 1)

        candidates = image_index.get(path_key(rel, 2), [])
        candidates = [c for c in candidates if split in str(c).lower()] or candidates
        if not candidates:
            continue

        boxes = []
        for line in box_lines[:count]:
            v = line.split()
            if len(v) < 4:
                continue
            x, y, w, h = map(float, v[:4])
            invalid = len(v) >= 8 and v[7] == "1"
            if w > 0 and h > 0 and not invalid:
                boxes.append([x, y, x + w, y + h])
        samples.append({"load": file_loader(candidates[0]), "boxes": boxes,
                        "image_id": f"wider_face:{rel}"})
    return samples, {"annotation_file": str(gt_file), "split": split}


# ------------------------------------------------------------
# FDDB  (FDDB-fold-XX-ellipseList.txt, ellipses -> axis-aligned boxes)
# ------------------------------------------------------------
def collect_fddb(root):
    root = Path(root)
    ellipse_files = sorted(p for p in root.rglob("*.txt") if "ellipselist" in p.name.lower())
    if not ellipse_files:
        raise FileNotFoundError("FDDB *-ellipseList.txt annotation files not found")

    # FDDB file names repeat across days (img_130.jpg exists many times), so match on
    # the full relative path "2002/07/19/big/img_130".
    image_index = {path_key(p, 5): p for p in list_images(root)}

    samples, seen = [], set()
    for ann in ellipse_files:
        lines = ann.read_text(encoding="utf-8", errors="replace").splitlines()
        i = 0
        while i < len(lines):
            rel = lines[i].strip(); i += 1
            if not rel or rel.replace(".", "").isdigit():
                continue
            try:
                count = int(lines[i].strip()); i += 1
            except (ValueError, IndexError):
                continue
            ellipse_lines = lines[i:i + count]; i += count

            key = path_key(rel, 5)
            if key in seen or key not in image_index:
                continue
            seen.add(key)

            boxes = []
            for line in ellipse_lines:
                v = line.split()
                if len(v) < 5:
                    continue
                major, minor, angle, cx, cy = map(float, v[:5])
                rx = math.sqrt((major * math.cos(angle)) ** 2 + (minor * math.sin(angle)) ** 2)
                ry = math.sqrt((major * math.sin(angle)) ** 2 + (minor * math.cos(angle)) ** 2)
                boxes.append([cx - rx, cy - ry, cx + rx, cy + ry])
            samples.append({"load": file_loader(image_index[key]), "boxes": boxes,
                            "image_id": f"fddb:{rel}"})
    return samples, {"annotation_files": len(ellipse_files)}


# ------------------------------------------------------------
# CelebA aligned.  list_bbox_celeba.csv refers to the ORIGINAL in-the-wild
# photos, not the 178x218 aligned crops shipped on Kaggle, so those boxes
# are unusable here. The aligned 5-point landmarks DO match the crops, so
# matching uses landmark containment and the GT box is a landmark-derived
# approximation (used for drawing and face-size statistics only).
# ------------------------------------------------------------
def landmarks_to_box(points):
    """Approximate face box from 5 landmarks (eyes, nose, mouth corners)."""
    p = np.asarray(points, dtype=np.float32)
    eye_center = (p[0] + p[1]) / 2
    mouth_center = (p[3] + p[4]) / 2
    eye_dist = max(np.linalg.norm(p[1] - p[0]), 1.0)
    cx = (eye_center[0] + mouth_center[0]) / 2
    return [cx - 1.15 * eye_dist, eye_center[1] - 1.1 * eye_dist,
            cx + 1.15 * eye_dist, mouth_center[1] + 0.85 * eye_dist]


def collect_celeba(root):
    root = Path(root)
    lm_csv = next((p for p in root.rglob("*.csv") if p.name.lower() == "list_landmarks_align_celeba.csv"), None)
    if lm_csv is None:
        raise FileNotFoundError("list_landmarks_align_celeba.csv not found")
    table = pd.read_csv(lm_csv, skipinitialspace=True)
    table.columns = [c.strip() for c in table.columns]
    cols = ["lefteye_x", "lefteye_y", "righteye_x", "righteye_y", "nose_x", "nose_y",
            "leftmouth_x", "leftmouth_y", "rightmouth_x", "rightmouth_y"]
    if not set(cols + ["image_id"]).issubset(table.columns):
        raise ValueError(f"Unexpected CelebA landmark columns: {list(table.columns)}")

    # Prefer the official test partition when available.
    part_csv = next((p for p in root.rglob("*.csv") if p.name.lower() == "list_eval_partition.csv"), None)
    if part_csv is not None:
        part = pd.read_csv(part_csv)
        part.columns = [c.strip() for c in part.columns]
        test_ids = set(part.loc[part["partition"] == 2, "image_id"])
        if test_ids:
            table = table[table["image_id"].isin(test_ids)]

    image_index = {p.name.lower(): p for p in list_images(root)}
    zip_path = next((p for p in root.rglob("img_align_celeba.zip")), None)
    zip_index = {}
    if not image_index and zip_path is not None:
        zip_index = {Path(n).name.lower(): n for n in _open_zip(str(zip_path)).namelist()
                     if not n.endswith("/")}
    if not image_index and not zip_index:
        raise FileNotFoundError("CelebA aligned images (folder or img_align_celeba.zip) not found")

    samples = []
    for row in table.itertuples(index=False):
        name = str(row.image_id).strip()
        pts = np.array([getattr(row, c) for c in cols], dtype=np.float32).reshape(1, 5, 2)
        if name.lower() in image_index:
            loader = file_loader(image_index[name.lower()])
        elif name.lower() in zip_index:
            loader = zip_loader(zip_path, zip_index[name.lower()])
        else:
            continue
        samples.append({"load": loader, "boxes": [landmarks_to_box(pts[0])],
                        "landmarks": pts, "image_id": f"celeba:{name}"})
    return samples, {"partition": "test" if part_csv is not None else "all",
                     "note": "GT boxes approximated from aligned landmarks; matching uses landmarks"}


# ------------------------------------------------------------
# Pascal VOC XML
# ------------------------------------------------------------
def collect_voc(root, label_filter=None):
    root = Path(root)
    xml_files = sorted(root.rglob("*.xml"))
    if not xml_files:
        raise FileNotFoundError("No Pascal VOC XML files found")
    by_name = {p.name.lower(): p for p in list_images(root)}
    by_stem = {p.stem.lower(): p for p in list_images(root)}

    samples = []
    for xml_path in xml_files:
        try:
            node = ET.parse(xml_path).getroot()
        except ET.ParseError:
            continue
        if node.tag != "annotation":
            continue
        filename = (node.findtext("filename") or "").strip()
        image_path = by_name.get(Path(filename).name.lower()) or by_stem.get(xml_path.stem.lower())
        if image_path is None:
            continue
        boxes = []
        for obj in node.findall("object"):
            label = (obj.findtext("name") or "").strip().lower()
            if label_filter and label not in label_filter:
                continue
            bb = obj.find("bndbox")
            try:
                boxes.append([float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")])
            except (AttributeError, TypeError, ValueError):
                continue
        samples.append({"load": file_loader(image_path), "boxes": boxes,
                        "image_id": f"voc:{image_path.name}"})
    if not samples:
        raise ValueError("VOC XML files found but none matched an image")
    return samples, {"xml_files": len(xml_files)}


# ------------------------------------------------------------
# YOLO txt  (class cx cy w h, normalised)
# ------------------------------------------------------------
def _parse_yolo_file(path):
    rows = []
    for line in path.read_text(errors="replace").splitlines():
        v = line.split()
        if not v:
            continue
        if len(v) < 5:
            return None
        try:
            nums = [float(x) for x in v[1:5]]
        except ValueError:
            return None
        if any(n < -0.01 or n > 1.01 for n in nums):
            return None          # not normalised YOLO (e.g. absolute-pixel label files)
        rows.append(nums)
    return rows


def collect_yolo(root):
    root = Path(root)
    images = list_images(root)
    label_files = [p for p in root.rglob("*.txt")
                   if "label" in str(p.parent).lower() and p.name.lower() not in {"classes.txt", "readme.txt"}]
    labels = {}
    for p in prefer_split(label_files):
        parsed = _parse_yolo_file(p)
        if parsed is not None:
            labels.setdefault(p.stem.lower(), parsed)
    if not labels:
        raise FileNotFoundError("No normalised YOLO label files found")

    samples = []
    for img in prefer_split(images):
        rows = labels.get(img.stem.lower())
        if rows is None:
            continue

        def load(path=img):
            return Image.open(path)

        # Boxes are converted to pixels lazily because the image size is needed.
        def boxes_for(rows=rows, path=img):
            with Image.open(path) as im:
                w, h = im.size
            return [[(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h]
                    for cx, cy, bw, bh in rows]

        samples.append({"load": load, "boxes_fn": boxes_for, "boxes": None,
                        "image_id": f"yolo:{img.parent.name}/{img.name}"})
    if not samples:
        raise ValueError("YOLO labels found but none matched an image")
    return samples, {"label_files": len(labels)}


# ------------------------------------------------------------
# COCO json
# ------------------------------------------------------------
def collect_coco(root):
    root = Path(root)
    by_name = {p.name.lower(): p for p in list_images(root)}
    for json_path in prefer_split(root.rglob("*.json")):
        try:
            data = json.loads(json_path.read_text(errors="replace"))
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue
        if not (isinstance(data, dict) and "images" in data and "annotations" in data):
            continue
        boxes = {}
        for a in data["annotations"]:
            x, y, w, h = a["bbox"]
            boxes.setdefault(a["image_id"], []).append([x, y, x + w, y + h])
        samples = []
        for im in data["images"]:
            path = by_name.get(Path(im["file_name"]).name.lower())
            if path is not None:
                samples.append({"load": file_loader(path), "boxes": boxes.get(im["id"], []),
                                "image_id": f"coco:{path.name}"})
        if samples:
            return samples, {"annotation_file": str(json_path)}
    raise FileNotFoundError("No COCO-format json found")


def collect_auto(root):
    """Try VOC, then COCO, then YOLO; report every failure if none work."""
    errors = []
    for fmt, fn in (("voc", collect_voc), ("coco", collect_coco), ("yolo", collect_yolo)):
        try:
            samples, meta = fn(root)
            meta["detected_format"] = fmt
            return samples, meta
        except Exception as exc:
            errors.append(f"{fmt}: {exc}")
    raise ValueError("Could not detect annotation format -> " + " | ".join(errors))


COLLECTORS = {
    "wider": collect_wider_face,
    "fddb": collect_fddb,
    "celeba": collect_celeba,
    "voc": collect_voc,
    "yolo": collect_yolo,
    "coco": collect_coco,
    "auto": collect_auto,
}


def build_dataset(config, root, preprocess_fn=preprocess_face_sample,
                  limit=MAX_IMAGES_PER_DATASET, seed=RANDOM_SEED):
    """Collect annotations -> sample -> preprocess -> FaceDataset."""
    samples, meta = COLLECTORS[config["builder"]](root)
    total = len(samples)

    rng = np.random.default_rng(seed)
    if limit is not None and total > limit:
        samples = [samples[i] for i in sorted(rng.choice(total, size=limit, replace=False))]
    for s in samples:                      # resolve lazily-computed boxes (YOLO)
        if s.get("boxes") is None and "boxes_fn" in s:
            s["boxes"] = s.pop("boxes_fn")()

    meta.update({"kaggle_handle": config["kaggle_handle"],
                 "description": config.get("description", ""),
                 "available_annotated_images": total})
    return make_face_dataset(
        config["name"], samples, preprocess_fn=preprocess_fn,
        match_mode="landmarks" if config["builder"] == "celeba" else "iou",
        limit=None, seed=seed, metadata=meta,
    )


def build_custom_dataset(name, image_paths, annotation_map, preprocess_fn=preprocess_face_sample):
    """
    annotation_map: {image_basename: [[x1, y1, x2, y2], ...]}  ([] for images with no faces)
    """
    samples = []
    for path in map(Path, image_paths):
        if path.name not in annotation_map:
            raise ValueError(f"Missing annotation for {path.name}; use [] for images without faces")
        samples.append({"load": file_loader(path), "boxes": annotation_map[path.name],
                        "image_id": path.name})
    return make_face_dataset(name, samples, preprocess_fn=preprocess_fn)


## Kaggle download and verification

In [ ]:
# ============================================================
# KAGGLE DOWNLOAD + RUNTIME VERIFICATION
# ============================================================

def download_kaggle_dataset(handle):
    destination = DOWNLOAD_DIR / handle.replace("/", "__")
    try:
        path = kagglehub.dataset_download(handle, output_dir=str(destination))
    except TypeError:                     # older kagglehub without output_dir
        path = kagglehub.dataset_download(handle)
    return Path(path)


def verify_kaggle_datasets(configs):
    """
    A dataset counts as verified only if (1) it downloads from Kaggle and
    (2) its annotations parse into at least one annotated image.
    """
    rows, roots = [], {}
    for cfg in configs:
        row = {"dataset": cfg["name"], "kaggle_handle": cfg["kaggle_handle"],
               "kaggle_url": f"https://www.kaggle.com/datasets/{cfg['kaggle_handle']}",
               "downloaded": False, "annotations_parsed": False,
               "annotated_images": 0, "verified": False, "error": ""}
        try:
            root = download_kaggle_dataset(cfg["kaggle_handle"])
            row["downloaded"] = True
            row["files"] = sum(1 for p in root.rglob("*") if p.is_file())
            samples, meta = COLLECTORS[cfg["builder"]](root)
            row["annotations_parsed"] = True
            row["annotated_images"] = len(samples)
            row["detected_format"] = meta.get("detected_format", cfg["builder"])
            row["verified"] = len(samples) > 0
            roots[cfg["name"]] = root
        except Exception as exc:
            row["error"] = repr(exc)[:500]
        rows.append(row)
        status = "VERIFIED" if row["verified"] else "FAILED"
        print(f"[{status}] {cfg['name']:<22} {cfg['kaggle_handle']}  "
              f"images={row['annotated_images']}  {row['error']}")
    report = pd.DataFrame(rows)
    report.to_csv(OUTPUT_DIR / "dataset_verification.csv", index=False)
    return report, roots


## Matching and metric utilities

In [ ]:
# ============================================================
# MATCHING & METRIC UTILITIES
# ============================================================

def iou_matrix(pred, gt):
    pred = np.asarray(pred, dtype=np.float32).reshape(-1, 4)
    gt = np.asarray(gt, dtype=np.float32).reshape(-1, 4)
    if len(pred) == 0 or len(gt) == 0:
        return np.zeros((len(pred), len(gt)), dtype=np.float32)
    x1 = np.maximum(pred[:, None, 0], gt[None, :, 0])
    y1 = np.maximum(pred[:, None, 1], gt[None, :, 1])
    x2 = np.minimum(pred[:, None, 2], gt[None, :, 2])
    y2 = np.minimum(pred[:, None, 3], gt[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_p = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    area_g = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    union = area_p[:, None] + area_g[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0).astype(np.float32)


def landmark_match_matrix(pred, gt_boxes, gt_landmarks, min_iou=0.2):
    """Fraction of a face's 5 landmarks inside each predicted box (0 if box is wildly off)."""
    pred = np.asarray(pred, dtype=np.float32).reshape(-1, 4)
    if len(pred) == 0 or len(gt_landmarks) == 0:
        return np.zeros((len(pred), len(gt_landmarks)), dtype=np.float32)
    lm = np.asarray(gt_landmarks, dtype=np.float32)              # (G, 5, 2)
    inside = ((lm[None, :, :, 0] >= pred[:, None, None, 0]) & (lm[None, :, :, 0] <= pred[:, None, None, 2]) &
              (lm[None, :, :, 1] >= pred[:, None, None, 1]) & (lm[None, :, :, 1] <= pred[:, None, None, 3]))
    frac = inside.mean(axis=2)
    sanity = iou_matrix(pred, gt_boxes) >= min_iou
    return (frac * sanity).astype(np.float32)


def greedy_match(scores, sim, threshold):
    """Highest-score detection claims its best unmatched GT if sim >= threshold."""
    P, G = sim.shape
    tp = np.zeros(P, dtype=bool)
    matched_gt = np.full(P, -1, dtype=int)
    matched_sim = np.zeros(P, dtype=np.float32)
    if P == 0 or G == 0:
        return tp, matched_gt, matched_sim
    used = np.zeros(G, dtype=bool)
    for p in np.argsort(-np.asarray(scores), kind="stable"):
        candidates = np.where(used, -1.0, sim[p])
        g = int(np.argmax(candidates))
        if candidates[g] >= threshold - 1e-9:
            tp[p], matched_gt[p], matched_sim[p] = True, g, sim[p, g]
            used[g] = True
    return tp, matched_gt, matched_sim


def pr_curve(scores, tp_flags, n_gt):
    order = np.argsort(-np.asarray(scores), kind="stable")
    tp = np.asarray(tp_flags, dtype=float)[order]
    s = np.asarray(scores, dtype=float)[order]
    tp_cum, fp_cum = np.cumsum(tp), np.cumsum(1 - tp)
    recall = tp_cum / max(n_gt, 1)
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1)
    return s, precision, recall, tp_cum, fp_cum


def average_precision(precision, recall):
    """VOC2010+/COCO-style all-point interpolated AP."""
    if len(precision) == 0:
        return 0.0
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[0.0], precision, [0.0]])
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def safe_div(a, b):
    return float(a) / float(b) if b else 0.0


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return None if not np.isfinite(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj


def save_json(obj, path):
    Path(path).write_text(json.dumps(to_jsonable(obj), indent=2), encoding="utf-8")


SIZE_BUCKETS = [("tiny (<16px)", 0, 16), ("small (16-32px)", 16, 32),
                ("medium (32-96px)", 32, 96), ("large (>=96px)", 96, float("inf"))]


## `RetinaFace` evaluation class

In [ ]:
# ============================================================
# RetinaFace EVALUATION CLASS
# ============================================================

@dataclass
class EvaluationResult:
    dataset_name: str
    match_mode: str
    metrics: Dict
    confusion: Dict
    curves: Dict
    per_image: pd.DataFrame
    detections: pd.DataFrame
    size_recall: pd.DataFrame
    dataset_summary: Dict
    visual_samples: List
    output_dir: Path
    figures: Dict = field(default_factory=dict)


class RetinaFace:
    """
    Wraps the retina-face detector and evaluates it on FaceDataset objects.

        detector = RetinaFace()
        result = detector.evaluate(dataset)          # metrics + CSV/JSON + figures
        detector.compare([result_a, result_b, ...])  # cross-dataset comparison
    """

    # Figures shown inline during evaluate(show=True); everything is always saved.
    INLINE_FIGURES = ("metrics_summary", "confusion_matrices", "roc_curves", "precision_recall")

    def __init__(self, score_threshold=SCORE_THRESHOLD, iou_threshold=IOU_THRESHOLD,
                 inference_threshold=INFERENCE_THRESHOLD, allow_upscaling=True,
                 output_dir=OUTPUT_DIR, dpi=FIGURE_DPI, model=None, warmup=True):
        if inference_threshold > score_threshold:
            raise ValueError("inference_threshold must be <= score_threshold")
        self.score_threshold = float(score_threshold)
        self.iou_threshold = float(iou_threshold)
        self.inference_threshold = float(inference_threshold)
        self.allow_upscaling = allow_upscaling
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.dpi = dpi
        self.model = model if model is not None else self._build_model()
        if warmup:  # first call builds the TF graph; keep it out of timing stats
            self.detect(np.zeros((320, 320, 3), dtype=np.uint8))

    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------
    def _build_model(self):
        return RetinaFaceLib.build_model()

    def _run_model(self, image_bgr):
        try:
            return RetinaFaceLib.detect_faces(image_bgr, threshold=self.inference_threshold,
                                              model=self.model, allow_upscaling=self.allow_upscaling)
        except TypeError:
            return RetinaFaceLib.detect_faces(image_bgr, threshold=self.inference_threshold,
                                              model=self.model)

    def detect(self, image):
        """image: RGB uint8 array. Returns [{"box": [x1,y1,x2,y2], "score": float, "landmarks": dict}]."""
        h, w = image.shape[:2]
        # retina-face follows the OpenCV convention and expects BGR input.
        raw = self._run_model(np.ascontiguousarray(image[:, :, ::-1]))
        detections = []
        if not isinstance(raw, dict):
            return detections
        for det in raw.values():
            box = det.get("facial_area")
            if box is None:
                continue
            x1, y1, x2, y2 = (float(v) for v in box)
            x1, x2 = np.clip([x1, x2], 0, w)
            y1, y2 = np.clip([y1, y2], 0, h)
            if x2 > x1 and y2 > y1:
                detections.append({"box": [float(x1), float(y1), float(x2), float(y2)],
                                   "score": float(det.get("score", 0.0)),
                                   "landmarks": det.get("landmarks")})
        return detections

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    def evaluate(self, dataset, save=True, plot=True, show=True,
                 num_visual_samples=NUM_VISUAL_SAMPLES):
        if not isinstance(dataset, FaceDataset):
            raise TypeError("evaluate() expects a FaceDataset")
        if len(dataset) == 0:
            raise ValueError(f"{dataset.name} is empty")

        landmark_mode = dataset.match_mode == "landmarks"
        match_threshold = 1.0 if landmark_mode else self.iou_threshold
        rng = np.random.default_rng(RANDOM_SEED)
        visual_index = set(rng.choice(len(dataset), size=min(num_visual_samples, len(dataset)),
                                      replace=False).tolist())

        per_image, det_rows, visual = [], [], []
        all_scores, all_tp, iou_cache = [], [], []
        gt_sizes, gt_found = [], []
        start = time.perf_counter()

        for idx, sample in enumerate(tqdm(dataset, total=len(dataset), desc=f"Evaluating {dataset.name}")):
            image, gt = sample["image"], sample["boxes"]
            t0 = time.perf_counter()
            preds = self.detect(image)
            seconds = time.perf_counter() - t0

            boxes = np.array([p["box"] for p in preds], dtype=np.float32).reshape(-1, 4)
            scores = np.array([p["score"] for p in preds], dtype=np.float32)
            iou = iou_matrix(boxes, gt)
            sim = landmark_match_matrix(boxes, gt, sample["landmarks"]) if landmark_mode else iou
            tp, matched_gt, _ = greedy_match(scores, sim, match_threshold)

            above = scores >= self.score_threshold
            tp_op = int((tp & above).sum())
            n_pred = int(above.sum())
            found = set(matched_gt[tp & above].tolist())
            sizes = np.sqrt(np.clip((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1]), 0, None))
            gt_sizes.extend(sizes.tolist())
            gt_found.extend(g in found for g in range(len(gt)))
            op_ious = iou[np.where(tp & above)[0], matched_gt[tp & above]] if tp_op else np.zeros(0)

            per_image.append({
                "dataset": dataset.name, "image_id": sample["image_id"],
                "width": image.shape[1], "height": image.shape[0],
                "gt_faces": len(gt), "predicted_faces": n_pred, "raw_detections": len(scores),
                "tp": tp_op, "fp": n_pred - tp_op, "fn": len(gt) - tp_op,
                "max_score": float(scores.max()) if len(scores) else 0.0,
                "mean_matched_iou": float(op_ious.mean()) if len(op_ious) else np.nan,
                "count_error": n_pred - len(gt),
                "inference_ms": seconds * 1000.0,
            })
            for p in range(len(scores)):
                det_rows.append({
                    "dataset": dataset.name, "image_id": sample["image_id"],
                    "score": float(scores[p]),
                    "x1": boxes[p, 0], "y1": boxes[p, 1], "x2": boxes[p, 2], "y2": boxes[p, 3],
                    "best_iou": float(iou[p].max()) if iou.shape[1] else 0.0,
                    "match_score": float(sim[p].max()) if sim.shape[1] else 0.0,
                    "is_tp": bool(tp[p]), "above_threshold": bool(above[p]),
                })
            all_scores.extend(scores.tolist())
            all_tp.extend(tp.tolist())
            iou_cache.append((scores, iou))
            if idx in visual_index:
                visual.append(self._visual_sample(image, gt, boxes[above], scores[above],
                                                  tp[above], sample["image_id"]))

        elapsed = time.perf_counter() - start
        per_image_df = pd.DataFrame(per_image)
        detections_df = pd.DataFrame(det_rows, columns=[
            "dataset", "image_id", "score", "x1", "y1", "x2", "y2",
            "best_iou", "match_score", "is_tp", "above_threshold"])

        metrics, confusion, curves, size_df = self._compute_metrics(
            dataset, per_image_df, np.array(all_scores, dtype=float), np.array(all_tp, dtype=bool),
            iou_cache, np.array(gt_sizes), np.array(gt_found, dtype=bool), landmark_mode, elapsed)

        result = EvaluationResult(
            dataset_name=dataset.name, match_mode=dataset.match_mode, metrics=metrics,
            confusion=confusion, curves=curves, per_image=per_image_df, detections=detections_df,
            size_recall=size_df, dataset_summary={**dataset.summary(), **dataset.metadata},
            visual_samples=visual, output_dir=self.output_dir / dataset.name)

        if save:
            self.save_results(result)
        if plot:
            self.plot_results(result, show=show)
        self._print_summary(result)
        return result

    # --------------------------------------------------------
    def _compute_metrics(self, dataset, per_image, scores, tp_flags, iou_cache,
                         gt_sizes, gt_found, landmark_mode, elapsed):
        n_images = len(per_image)
        n_gt = int(per_image["gt_faces"].sum())
        TP, FP, FN = (int(per_image[c].sum()) for c in ("tp", "fp", "fn"))

        precision = safe_div(TP, TP + FP)
        recall = safe_div(TP, TP + FN)
        f1 = safe_div(2 * precision * recall, precision + recall)

        # ---- PR / AP -------------------------------------------------
        s_sorted, prec_curve, rec_curve, tp_cum, fp_cum = pr_curve(scores, tp_flags, n_gt)
        ap_primary = average_precision(prec_curve, rec_curve)
        ap_by_iou = {}
        if not landmark_mode:
            for t in np.round(np.linspace(0.5, 0.95, 10), 2):
                sc, fl = [], []
                for s, iou in iou_cache:
                    tp_t, _, _ = greedy_match(s, iou, t)
                    sc.extend(s.tolist()); fl.extend(tp_t.tolist())
                _, p_t, r_t, _, _ = pr_curve(np.array(sc), np.array(fl, dtype=bool), n_gt)
                ap_by_iou[float(t)] = average_precision(p_t, r_t)

        # ---- Detection-level ROC (TP vs FP detections ranked by confidence)
        det_roc = {"fpr": [], "tpr": [], "thresholds": []}
        det_auc = float("nan")
        if tp_flags.any() and (~tp_flags).any():
            fpr, tpr, thr = roc_curve(tp_flags.astype(int), scores)
            det_roc = {"fpr": fpr, "tpr": tpr, "thresholds": thr}
            det_auc = float(roc_auc_score(tp_flags.astype(int), scores))

        # ---- Threshold sweep -------------------------------------------
        thresholds = np.round(np.linspace(self.inference_threshold, 0.999, 200), 4)
        asc = np.sort(scores)
        n_above = len(asc) - np.searchsorted(asc, thresholds, side="left")
        tp_above = np.where(n_above > 0, tp_cum[np.maximum(n_above - 1, 0)] if len(tp_cum) else 0, 0)
        sweep_p = np.where(n_above > 0, tp_above / np.maximum(n_above, 1), 0.0)
        sweep_r = tp_above / max(n_gt, 1)
        sweep_f1 = np.where(sweep_p + sweep_r > 0, 2 * sweep_p * sweep_r / np.maximum(sweep_p + sweep_r, 1e-12), 0.0)
        best = int(np.argmax(sweep_f1)) if len(sweep_f1) else 0

        # ---- Image-level (does the image contain a face?) ---------------
        y_true = (per_image["gt_faces"] > 0).astype(int).values
        y_score = per_image["max_score"].values
        y_pred = (per_image["predicted_faces"] > 0).astype(int).values
        tn, fp_i, fn_i, tp_i = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        img_roc = {"fpr": [], "tpr": [], "thresholds": []}
        img_auc = float("nan")
        if 0 < y_true.sum() < len(y_true):
            fpr_i, tpr_i, thr_i = roc_curve(y_true, y_score)
            img_roc = {"fpr": fpr_i, "tpr": tpr_i, "thresholds": thr_i}
            img_auc = float(roc_auc_score(y_true, y_score))
        img_precision = safe_div(tp_i, tp_i + fp_i)
        img_recall = safe_div(tp_i, tp_i + fn_i)

        # ---- Face size buckets ------------------------------------------
        size_rows = []
        for label, lo, hi in SIZE_BUCKETS:
            mask = (gt_sizes >= lo) & (gt_sizes < hi)
            size_rows.append({"dataset": dataset.name, "bucket": label, "faces": int(mask.sum()),
                              "detected": int(gt_found[mask].sum()),
                              "recall": safe_div(gt_found[mask].sum(), mask.sum()) if mask.any() else np.nan})
        size_df = pd.DataFrame(size_rows)

        times = per_image["inference_ms"].values
        matched_iou = per_image["mean_matched_iou"]

        metrics = {
            "dataset": dataset.name,
            "match_mode": dataset.match_mode,
            "images": n_images,
            "ground_truth_faces": n_gt,
            "predictions_at_threshold": TP + FP,
            "raw_detections": int(len(scores)),
            "true_positives": TP, "false_positives": FP, "false_negatives": FN,
            "true_negatives": None,  # undefined for detection (no countable background objects)
            "precision": precision, "recall": recall, "f1": f1,
            "detection_accuracy": safe_div(TP, TP + FP + FN),
            "miss_rate": safe_div(FN, TP + FN),
            "false_discovery_rate": safe_div(FP, TP + FP),
            "false_positives_per_image": safe_div(FP, n_images),
            "AP50": ap_by_iou.get(0.5, ap_primary) if not landmark_mode else float("nan"),
            "AP75": ap_by_iou.get(0.75, float("nan")),
            "mAP50_95": float(np.mean(list(ap_by_iou.values()))) if ap_by_iou else float("nan"),
            "AP_landmark_match": ap_primary if landmark_mode else float("nan"),
            "AP_primary": ap_primary,
            "detection_roc_auc": det_auc,
            "best_f1": float(sweep_f1[best]) if len(sweep_f1) else 0.0,
            "best_f1_threshold": float(thresholds[best]) if len(thresholds) else float("nan"),
            "mean_matched_iou": float(matched_iou.mean()) if matched_iou.notna().any() else float("nan"),
            "image_accuracy": safe_div(tp_i + tn, len(y_true)),
            "image_precision": img_precision,
            "image_recall": img_recall,
            "image_specificity": safe_div(tn, tn + fp_i) if (tn + fp_i) else float("nan"),
            "image_f1": safe_div(2 * img_precision * img_recall, img_precision + img_recall),
            "image_roc_auc": img_auc,
            "count_accuracy": float((per_image["count_error"] == 0).mean()),
            "count_within_1": float((per_image["count_error"].abs() <= 1).mean()),
            "count_mae": float(per_image["count_error"].abs().mean()),
            "mean_inference_ms": float(times.mean()),
            "median_inference_ms": float(np.median(times)),
            "p95_inference_ms": float(np.percentile(times, 95)),
            "images_per_second": safe_div(1000.0, times.mean()),
            "elapsed_seconds": elapsed,
            "score_threshold": self.score_threshold,
            "inference_threshold": self.inference_threshold,
            "iou_threshold": self.iou_threshold if not landmark_mode else None,
            "match_criterion": "all 5 landmarks inside box" if landmark_mode else f"IoU >= {self.iou_threshold}",
        }
        for label, _, _ in SIZE_BUCKETS:
            row = size_df[size_df["bucket"] == label].iloc[0]
            metrics[f"recall_{label.split()[0]}"] = row["recall"]

        confusion = {
            "detection": {"TP": TP, "FP": FP, "FN": FN, "TN": None,
                          "layout": "rows=actual [face, background], cols=predicted [face, background]"},
            "image_level": {"TP": int(tp_i), "FP": int(fp_i), "FN": int(fn_i), "TN": int(tn),
                            "layout": "rows=actual [has face, no face], cols=predicted [face, no face]"},
        }
        curves = {
            "pr": {"scores": s_sorted, "precision": prec_curve, "recall": rec_curve},
            "froc": {"fppi": fp_cum / max(n_images, 1), "false_positives": fp_cum, "recall": rec_curve},
            "detection_roc": det_roc,
            "image_roc": img_roc,
            "threshold_sweep": {"threshold": thresholds, "precision": sweep_p, "recall": sweep_r, "f1": sweep_f1},
            "ap_by_iou": ap_by_iou,
            "operating_point": {"precision": precision, "recall": recall,
                                "fppi": safe_div(FP, n_images),
                                "det_fpr": safe_div(((scores >= self.score_threshold) & ~tp_flags).sum(), (~tp_flags).sum()),
                                "det_tpr": safe_div(((scores >= self.score_threshold) & tp_flags).sum(), tp_flags.sum())},
        }
        return metrics, confusion, curves, size_df

    # --------------------------------------------------------
    # Saving
    # --------------------------------------------------------
    def save_results(self, result):
        out = Path(result.output_dir)
        out.mkdir(parents=True, exist_ok=True)
        c = result.curves
        save_json({"metrics": result.metrics, "confusion_matrices": result.confusion,
                   "dataset_summary": result.dataset_summary,
                   "ap_by_iou": c["ap_by_iou"]}, out / "metrics.json")
        save_json(result.confusion, out / "confusion_matrices.json")
        pd.DataFrame([result.metrics]).to_csv(out / "metrics.csv", index=False)
        result.per_image.to_csv(out / "per_image_metrics.csv", index=False)
        result.detections.to_csv(out / "detections.csv", index=False)
        result.size_recall.to_csv(out / "recall_by_face_size.csv", index=False)
        pd.DataFrame(c["pr"]).to_csv(out / "pr_curve.csv", index=False)
        pd.DataFrame(c["froc"]).to_csv(out / "froc_curve.csv", index=False)
        pd.DataFrame(c["detection_roc"]).to_csv(out / "roc_curve_detection.csv", index=False)
        pd.DataFrame(c["image_roc"]).to_csv(out / "roc_curve_image.csv", index=False)
        pd.DataFrame(c["threshold_sweep"]).to_csv(out / "threshold_sweep.csv", index=False)
        pd.DataFrame(list(c["ap_by_iou"].items()), columns=["iou_threshold", "AP"]).to_csv(out / "ap_by_iou.csv", index=False)
        return out

    # --------------------------------------------------------
    # Plotting (per dataset)
    # --------------------------------------------------------
    def _save_fig(self, fig, path, show=False):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=self.dpi, bbox_inches="tight", facecolor="white")
        if show:
            plt.show()
        plt.close(fig)
        return path

    @staticmethod
    def _draw_matrix(ax, values, row_labels, col_labels, title, notes=None, cmap="Blues", percent=False):
        arr = np.array([[np.nan if v is None else float(v) for v in row] for row in values])
        finite = np.isfinite(arr).any()
        vmax = float(np.nanmax(arr)) if finite else 1.0
        ax.imshow(np.ma.masked_invalid(arr), cmap=cmap, vmin=0, vmax=max(vmax, 1e-9))
        ax.set_xticks(range(len(col_labels)), col_labels)
        ax.set_yticks(range(len(row_labels)), row_labels)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.grid(False)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                v = values[i][j]
                if v is None:
                    text = "N/A"
                else:
                    text = f"{float(v):.1%}" if percent else f"{int(v):,}"
                if notes is not None and notes[i][j]:
                    text += f"\n{notes[i][j]}"
                dark = v is not None and vmax > 0 and float(v) > 0.6 * vmax
                ax.text(j, i, text, ha="center", va="center", fontsize=11,
                        color="white" if dark else "black", fontweight="bold")

    def plot_results(self, result, show=True):
        fig_dir = Path(result.output_dir) / "figures"
        figs = {}
        m, c, name = result.metrics, result.curves, result.dataset_name
        crit = m["match_criterion"]

        def done(key, fig):
            figs[key] = self._save_fig(fig, fig_dir / f"{key}.png", show=show and key in self.INLINE_FIGURES)

        # 1. Metrics summary -------------------------------------------------
        fig, (ax, ax_t) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.6, 1]})
        keys = ["precision", "recall", "f1", "detection_accuracy", "AP_primary", "mAP50_95",
                "detection_roc_auc", "best_f1", "image_accuracy", "image_roc_auc",
                "count_accuracy", "mean_matched_iou"]
        labels = ["Precision", "Recall", "F1", "Detection acc.", "AP (primary)", "mAP@.5:.95",
                  "Det. ROC-AUC", "Best F1", "Image acc.", "Image ROC-AUC", "Count acc.", "Mean IoU"]
        vals = [m.get(k) for k in keys]
        plot_vals = [v if v is not None and np.isfinite(v) else 0 for v in vals]
        bars = ax.barh(labels[::-1], plot_vals[::-1], color=plt.cm.viridis(np.linspace(0.2, 0.85, len(keys)))[::-1])
        for b, v in zip(bars, vals[::-1]):
            ax.text(b.get_width() + 0.01, b.get_y() + b.get_height() / 2,
                    "n/a" if v is None or not np.isfinite(v) else f"{v:.3f}", va="center", fontsize=9)
        ax.set_xlim(0, 1.12); ax.set_title(f"{name}: key metrics", fontweight="bold")
        ax_t.axis("off")
        info = [["Images", f"{m['images']:,}"], ["GT faces", f"{m['ground_truth_faces']:,}"],
                ["TP / FP / FN", f"{m['true_positives']:,} / {m['false_positives']:,} / {m['false_negatives']:,}"],
                ["Score threshold", f"{m['score_threshold']:.2f}"], ["Match", crit],
                ["Best-F1 threshold", f"{m['best_f1_threshold']:.3f}"],
                ["FP per image", f"{m['false_positives_per_image']:.3f}"],
                ["Count MAE", f"{m['count_mae']:.3f}"],
                ["Mean / p95 time", f"{m['mean_inference_ms']:.1f} / {m['p95_inference_ms']:.1f} ms"],
                ["Throughput", f"{m['images_per_second']:.2f} img/s"]]
        table = ax_t.table(cellText=info, colLabels=["Item", "Value"], loc="center", cellLoc="left")
        table.auto_set_font_size(False); table.set_fontsize(9.5); table.scale(1, 1.6)
        done("metrics_summary", fig)

        # 2. Confusion matrices ------------------------------------------------
        d, im = result.confusion["detection"], result.confusion["image_level"]
        fig, axes = plt.subplots(1, 3, figsize=(17, 5))
        n_gt, n_pred = d["TP"] + d["FN"], d["TP"] + d["FP"]
        self._draw_matrix(axes[0], [[d["TP"], d["FN"]], [d["FP"], None]],
                          ["Face", "Background"], ["Face", "Background"],
                          "Detection-level (counts)",
                          notes=[[f"{safe_div(d['TP'], n_gt):.1%} of GT", f"{safe_div(d['FN'], n_gt):.1%} of GT"],
                                 [f"{safe_div(d['FP'], n_pred):.1%} of preds", "undefined"]])
        img_counts = [[im["TP"], im["FN"]], [im["FP"], im["TN"]]]
        self._draw_matrix(axes[1], img_counts, ["Has face", "No face"], ["Face", "No face"],
                          "Image-level (counts)", cmap="Blues")
        row_sums = [max(im["TP"] + im["FN"], 1), max(im["FP"] + im["TN"], 1)]
        norm = [[im["TP"] / row_sums[0], im["FN"] / row_sums[0]],
                [None if im["FP"] + im["TN"] == 0 else im["FP"] / row_sums[1],
                 None if im["FP"] + im["TN"] == 0 else im["TN"] / row_sums[1]]]
        self._draw_matrix(axes[2], norm, ["Has face", "No face"], ["Face", "No face"],
                          "Image-level (row-normalised)", cmap="Greens", percent=True)
        fig.suptitle(f"{name}: confusion matrices @ score >= {self.score_threshold:.2f}  ({crit})",
                     fontweight="bold")
        done("confusion_matrices", fig)

        # 3. ROC curves --------------------------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
        r = c["detection_roc"]
        if len(r["fpr"]):
            axes[0].plot(r["fpr"], r["tpr"], lw=2, color="tab:blue",
                         label=f"ROC (AUC = {m['detection_roc_auc']:.3f})")
            op = c["operating_point"]
            axes[0].scatter([op["det_fpr"]], [op["det_tpr"]], s=80, color="red", zorder=5,
                            label=f"score = {self.score_threshold:.2f}")
        else:
            axes[0].text(0.5, 0.5, "Not available:\nall detections are TP or all are FP",
                         ha="center", va="center", transform=axes[0].transAxes)
        axes[0].set_title("Detection-level ROC\n(correct vs false detections ranked by confidence)")
        r = c["image_roc"]
        if len(r["fpr"]):
            axes[1].plot(r["fpr"], r["tpr"], lw=2, color="tab:green",
                         label=f"ROC (AUC = {m['image_roc_auc']:.3f})")
        else:
            axes[1].text(0.5, 0.5, "Not available:\nevery image in this subset contains a face\n"
                         "(no negative images -> FPR undefined)",
                         ha="center", va="center", transform=axes[1].transAxes)
        axes[1].set_title("Image-level ROC\n(face present vs absent, max detection score)")
        for ax in axes:
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="chance")
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ax.legend(loc="lower right")
        fig.suptitle(f"{name}: ROC curves", fontweight="bold")
        done("roc_curves", fig)

        # 4. Precision-recall + AP vs IoU ------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
        ax = axes[0]
        for f in (0.2, 0.4, 0.6, 0.8):
            x = np.linspace(0.01, 1, 200); y = f * x / (2 * x - f)
            ok = (y > 0) & (y <= 1.05)
            ax.plot(x[ok], y[ok], color="gray", alpha=0.25, lw=1)
            ax.annotate(f"F1={f}", xy=(0.92, f * 0.92 / (2 * 0.92 - f)), fontsize=7, color="gray")
        ax.plot(c["pr"]["recall"], c["pr"]["precision"], lw=2, color="tab:purple",
                label=f"PR curve (AP = {m['AP_primary']:.3f})")
        ax.scatter([m["recall"]], [m["precision"]], color="red", s=80, zorder=5,
                   label=f"score = {self.score_threshold:.2f} (F1 = {m['f1']:.3f})")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(loc="lower left")
        ax.set_title(f"Precision-Recall ({crit})")
        ax = axes[1]
        if c["ap_by_iou"]:
            ts, aps = list(c["ap_by_iou"].keys()), list(c["ap_by_iou"].values())
            ax.bar([f"{t:.2f}" for t in ts], aps, color=plt.cm.plasma(np.linspace(0.15, 0.85, len(ts))))
            for i, v in enumerate(aps):
                ax.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
            ax.axhline(m["mAP50_95"], color="black", ls="--", lw=1, label=f"mAP@.5:.95 = {m['mAP50_95']:.3f}")
            ax.legend(); ax.set_ylim(0, 1.05)
            ax.set_xlabel("IoU threshold"); ax.set_ylabel("AP")
        else:
            ax.text(0.5, 0.5, "AP over IoU thresholds not applicable\n(landmark-based matching)",
                    ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Average precision vs IoU threshold")
        fig.suptitle(f"{name}: precision / recall", fontweight="bold")
        done("precision_recall", fig)

        # 5. Threshold sweep -------------------------------------------------
        sw = c["threshold_sweep"]
        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.plot(sw["threshold"], sw["precision"], label="Precision", lw=2)
        ax.plot(sw["threshold"], sw["recall"], label="Recall", lw=2)
        ax.plot(sw["threshold"], sw["f1"], label="F1", lw=2.5, color="black")
        ax.axvline(self.score_threshold, color="red", ls="--", label=f"operating threshold {self.score_threshold:.2f}")
        ax.axvline(m["best_f1_threshold"], color="green", ls=":", lw=2,
                   label=f"best F1 {m['best_f1']:.3f} @ {m['best_f1_threshold']:.3f}")
        ax.set_xlabel("Confidence threshold"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.02)
        ax.set_title(f"{name}: metrics vs confidence threshold", fontweight="bold"); ax.legend()
        done("threshold_sweep", fig)

        # 6. FROC (FDDB-style discrete ROC) ---------------------------------
        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.plot(c["froc"]["fppi"], c["froc"]["recall"], lw=2, color="tab:orange")
        ax.scatter([c["operating_point"]["fppi"]], [m["recall"]], color="red", s=80, zorder=5,
                   label=f"score = {self.score_threshold:.2f}")
        ax.set_xscale("symlog", linthresh=0.01)
        ax.set_xlabel("False positives per image"); ax.set_ylabel("Recall (true positive rate)")
        ax.set_ylim(0, 1.02); ax.legend()
        ax.set_title(f"{name}: FROC curve (recall vs false positives per image)", fontweight="bold")
        done("froc_curve", fig)

        # 7. Score & IoU distributions ---------------------------------------
        det = result.detections
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        bins = np.linspace(self.inference_threshold, 1, 40)
        axes[0].hist(det.loc[det.is_tp, "score"], bins=bins, alpha=0.7, label="True positives", color="tab:blue")
        axes[0].hist(det.loc[~det.is_tp, "score"], bins=bins, alpha=0.7, label="False positives", color="tab:red")
        axes[0].axvline(self.score_threshold, color="black", ls="--", label="operating threshold")
        axes[0].set_yscale("log"); axes[0].set_xlabel("Confidence"); axes[0].set_ylabel("Detections (log)")
        axes[0].set_title("Confidence distribution"); axes[0].legend()
        above = det[det.above_threshold]
        axes[1].hist(above["best_iou"], bins=np.linspace(0, 1, 41), color="tab:cyan", edgecolor="white")
        axes[1].axvline(self.iou_threshold, color="red", ls="--", label=f"IoU threshold {self.iou_threshold}")
        axes[1].set_xlabel("Best IoU with any GT face"); axes[1].set_ylabel("Detections")
        axes[1].set_title(f"IoU distribution (detections >= {self.score_threshold:.2f})"); axes[1].legend()
        fig.suptitle(f"{name}: score & IoU distributions", fontweight="bold")
        done("score_iou_distributions", fig)

        # 8. Recall by face size ----------------------------------------------
        sr = result.size_recall
        fig, ax = plt.subplots(figsize=(10, 5))
        vals = sr["recall"].fillna(0).values
        bars = ax.bar(sr["bucket"], vals, color=plt.cm.cividis(np.linspace(0.2, 0.9, len(sr))))
        for b, (_, row) in zip(bars, sr.iterrows()):
            txt = "no faces" if row["faces"] == 0 else f"{row['recall']:.2f}\n({row['detected']}/{row['faces']})"
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, txt, ha="center", fontsize=9)
        ax.set_ylim(0, 1.15); ax.set_ylabel("Recall")
        ax.set_title(f"{name}: recall by face size (sqrt of box area, preprocessed pixels)", fontweight="bold")
        done("recall_by_face_size", fig)

        # 9. Face count analysis ----------------------------------------------
        pi = result.per_image
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        mx = max(pi["gt_faces"].max(), pi["predicted_faces"].max(), 1)
        jitter = np.random.default_rng(0).uniform(-0.2, 0.2, len(pi))
        axes[0].scatter(pi["gt_faces"] + jitter, pi["predicted_faces"] + jitter[::-1], alpha=0.4, s=18)
        axes[0].plot([0, mx], [0, mx], "r--", lw=1, label="perfect count")
        axes[0].set_xlabel("Ground-truth faces"); axes[0].set_ylabel("Predicted faces"); axes[0].legend()
        axes[0].set_title(f"Count accuracy = {m['count_accuracy']:.3f}, MAE = {m['count_mae']:.2f}")
        err = pi["count_error"].clip(-10, 10)
        axes[1].hist(err, bins=np.arange(-10.5, 11.5, 1), color="tab:olive", edgecolor="white")
        axes[1].set_xlabel("Predicted - GT faces (clipped to +/-10)"); axes[1].set_ylabel("Images")
        axes[1].set_title("Count error distribution")
        fig.suptitle(f"{name}: face counting", fontweight="bold")
        done("face_count_analysis", fig)

        # 10. Inference time -------------------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        axes[0].hist(pi["inference_ms"], bins=40, color="tab:gray", edgecolor="white")
        axes[0].axvline(m["mean_inference_ms"], color="red", ls="--", label=f"mean {m['mean_inference_ms']:.1f} ms")
        axes[0].axvline(m["p95_inference_ms"], color="purple", ls=":", label=f"p95 {m['p95_inference_ms']:.1f} ms")
        axes[0].set_xlabel("ms / image"); axes[0].set_ylabel("Images"); axes[0].legend()
        axes[0].set_title("Inference time distribution")
        axes[1].scatter(pi["width"] * pi["height"] / 1e6, pi["inference_ms"], alpha=0.4, s=15)
        axes[1].set_xlabel("Megapixels"); axes[1].set_ylabel("ms / image"); axes[1].set_title("Time vs resolution")
        fig.suptitle(f"{name}: speed ({m['images_per_second']:.2f} img/s)", fontweight="bold")
        done("inference_time", fig)

        # 11. Sample predictions ----------------------------------------------
        if result.visual_samples:
            n = len(result.visual_samples)
            cols = min(4, n); rows = math.ceil(n / cols)
            fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.2 * rows))
            axes = np.atleast_1d(axes).ravel()
            for ax in axes[n:]:
                ax.axis("off")
            for ax, vs in zip(axes, result.visual_samples):
                ax.imshow(vs["image"]); ax.axis("off"); ax.grid(False)
                for b in vs["gt"]:
                    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                                    fill=False, ec="lime", lw=1.6, ls="--"))
                for b, s, t in zip(vs["pred"], vs["scores"], vs["tp"]):
                    col = "deepskyblue" if t else "red"
                    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=col, lw=1.6))
                    ax.text(b[0], b[1] - 2, f"{s:.2f}", color="white", fontsize=7,
                            bbox=dict(facecolor=col, alpha=0.7, pad=1, lw=0))
                ax.set_title(f"{vs['image_id'][-38:]}\nGT {len(vs['gt'])} | pred {len(vs['pred'])}", fontsize=8)
            handles = [mpatches.Patch(ec="lime", fill=False, ls="--", label="Ground truth"),
                       mpatches.Patch(ec="deepskyblue", fill=False, label="True positive"),
                       mpatches.Patch(ec="red", fill=False, label="False positive")]
            fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)
            fig.suptitle(f"{name}: sample predictions @ score >= {self.score_threshold:.2f}", fontweight="bold")
            done("sample_predictions", fig)

        result.figures = figs
        return figs

    def _visual_sample(self, image, gt, boxes, scores, tp, image_id, max_side=640):
        scale = min(1.0, max_side / max(image.shape[:2]))
        if scale < 1.0:
            small = np.asarray(Image.fromarray(image).resize(
                (max(1, int(image.shape[1] * scale)), max(1, int(image.shape[0] * scale))), Image.BILINEAR))
        else:
            small = image.copy()
        return {"image": small, "gt": gt * scale, "pred": boxes * scale, "scores": scores,
                "tp": tp, "image_id": image_id}

    def _print_summary(self, result):
        m = result.metrics
        print(f"\n{result.dataset_name} ({m['images']} images, {m['ground_truth_faces']} faces, {m['match_criterion']})")
        print(f"  precision={m['precision']:.4f}  recall={m['recall']:.4f}  F1={m['f1']:.4f}  "
              f"det_acc={m['detection_accuracy']:.4f}")
        print(f"  AP={m['AP_primary']:.4f}  mAP50-95={m['mAP50_95']:.4f}  det_ROC-AUC={m['detection_roc_auc']:.4f}  "
              f"img_acc={m['image_accuracy']:.4f}  img_ROC-AUC={m['image_roc_auc']:.4f}")
        print(f"  {m['mean_inference_ms']:.1f} ms/img  ->  saved to {result.output_dir}")

    # --------------------------------------------------------
    # Cross-dataset comparison
    # --------------------------------------------------------
    COMPARISON_METRICS = [
        ("precision", "Precision"), ("recall", "Recall"), ("f1", "F1"),
        ("detection_accuracy", "Detection acc."), ("AP_primary", "AP"),
        ("mAP50_95", "mAP@.5:.95"), ("detection_roc_auc", "Det. ROC-AUC"),
        ("best_f1", "Best F1"), ("image_accuracy", "Image acc."),
        ("count_accuracy", "Count acc."), ("mean_matched_iou", "Mean IoU"),
    ]

    def compare(self, results, show=True, output_name="comparison"):
        results = [r for r in results if r is not None]
        if not results:
            raise ValueError("No results to compare")
        out = self.output_dir / output_name
        out.mkdir(parents=True, exist_ok=True)
        names = [r.dataset_name for r in results]
        colors = plt.cm.tab10(np.arange(len(results)) % 10)
        figs = {}

        def done(key, fig):
            figs[key] = self._save_fig(fig, out / f"{key}.png", show=show)

        # ---- tables -------------------------------------------------------
        table = pd.DataFrame([r.metrics for r in results])
        stats = pd.DataFrame([r.dataset_summary for r in results])
        size = pd.concat([r.size_recall for r in results], ignore_index=True)
        table.to_csv(out / "metrics_comparison.csv", index=False)
        stats.to_csv(out / "dataset_statistics.csv", index=False)
        size.to_csv(out / "recall_by_face_size_comparison.csv", index=False)
        save_json({"metrics": [r.metrics for r in results],
                   "dataset_statistics": [r.dataset_summary for r in results]}, out / "metrics_comparison.json")
        ranking = table.sort_values("f1", ascending=False)[
            ["dataset", "f1", "precision", "recall", "AP_primary", "detection_roc_auc", "images_per_second"]]
        ranking.insert(0, "rank_by_f1", range(1, len(ranking) + 1))
        ranking.to_csv(out / "ranking_by_f1.csv", index=False)

        keys = [k for k, _ in self.COMPARISON_METRICS]
        labels = [l for _, l in self.COMPARISON_METRICS]
        values = table[keys].astype(float).values

        # ---- grouped bar chart ------------------------------------------------
        fig, ax = plt.subplots(figsize=(max(14, 1.6 * len(keys)), 6.5))
        width = 0.8 / len(results)
        x = np.arange(len(keys))
        for i, (name, row) in enumerate(zip(names, values)):
            v = np.nan_to_num(row, nan=0.0)
            ax.bar(x + i * width - 0.4 + width / 2, v, width, label=name, color=colors[i])
        ax.set_xticks(x, labels, rotation=20); ax.set_ylim(0, 1.05)
        ax.set_title("RetinaFace: metric comparison across datasets (missing values shown as 0)", fontweight="bold")
        ax.legend(ncol=min(len(results), 3), loc="upper center", bbox_to_anchor=(0.5, -0.15))
        done("comparison_metrics_bar", fig)

        # ---- heatmap ----------------------------------------------------------
        fig, ax = plt.subplots(figsize=(max(12, 1.2 * len(keys)), 1 + 0.75 * len(results)))
        ax.imshow(np.ma.masked_invalid(values), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(keys)), labels, rotation=30, ha="right")
        ax.set_yticks(range(len(names)), names); ax.grid(False)
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                v = values[i, j]
                ax.text(j, i, "n/a" if not np.isfinite(v) else f"{v:.3f}", ha="center", va="center", fontsize=9)
        ax.set_title("Metric heatmap (green = better)", fontweight="bold")
        done("comparison_heatmap", fig)

        # ---- radar ------------------------------------------------------------
        radar_keys = ["precision", "recall", "f1", "detection_accuracy", "AP_primary", "detection_roc_auc", "count_accuracy"]
        radar_labels = ["Precision", "Recall", "F1", "Det. acc.", "AP", "Det. ROC-AUC", "Count acc."]
        angles = np.linspace(0, 2 * np.pi, len(radar_keys), endpoint=False).tolist() + [0]
        fig = plt.figure(figsize=(8.5, 8.5)); ax = fig.add_subplot(111, polar=True)
        for i, r in enumerate(results):
            v = [np.nan_to_num(float(r.metrics[k]) if r.metrics[k] is not None else 0.0) for k in radar_keys]
            v += v[:1]
            ax.plot(angles, v, lw=2, color=colors[i], label=r.dataset_name)
            ax.fill(angles, v, alpha=0.08, color=colors[i])
        ax.set_xticks(angles[:-1], radar_labels); ax.set_ylim(0, 1)
        ax.set_title("Dataset profile radar", fontweight="bold", pad=20)
        ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
        done("comparison_radar", fig)

        # ---- overlaid curves --------------------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for i, r in enumerate(results):
            axes[0].plot(r.curves["pr"]["recall"], r.curves["pr"]["precision"], lw=2, color=colors[i],
                         label=f"{r.dataset_name} (AP={r.metrics['AP_primary']:.3f})")
            axes[0].scatter([r.metrics["recall"]], [r.metrics["precision"]], color=colors[i], s=50, edgecolor="black", zorder=5)
            axes[1].plot(r.curves["froc"]["fppi"], r.curves["froc"]["recall"], lw=2, color=colors[i], label=r.dataset_name)
        axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)
        axes[0].set_title("Precision-recall (dots = operating threshold)"); axes[0].legend(fontsize=8, loc="lower left")
        axes[1].set_xscale("symlog", linthresh=0.01); axes[1].set_ylim(0, 1.02)
        axes[1].set_xlabel("False positives per image"); axes[1].set_ylabel("Recall")
        axes[1].set_title("FROC"); axes[1].legend(fontsize=8, loc="lower right")
        done("comparison_pr_froc_curves", fig)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        any_img_roc = False
        for i, r in enumerate(results):
            dr = r.curves["detection_roc"]
            if len(dr["fpr"]):
                axes[0].plot(dr["fpr"], dr["tpr"], lw=2, color=colors[i],
                             label=f"{r.dataset_name} (AUC={r.metrics['detection_roc_auc']:.3f})")
            ir = r.curves["image_roc"]
            if len(ir["fpr"]):
                any_img_roc = True
                axes[1].plot(ir["fpr"], ir["tpr"], lw=2, color=colors[i],
                             label=f"{r.dataset_name} (AUC={r.metrics['image_roc_auc']:.3f})")
        if not any_img_roc:
            axes[1].text(0.5, 0.5, "No dataset subset contains face-free images,\nso image-level ROC is undefined",
                         ha="center", va="center", transform=axes[1].transAxes)
        for ax, t in zip(axes, ["Detection-level ROC", "Image-level ROC"]):
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_title(t)
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=8, loc="lower right")
        done("comparison_roc_curves", fig)

        fig, ax = plt.subplots(figsize=(11, 6))
        for i, r in enumerate(results):
            sw = r.curves["threshold_sweep"]
            ax.plot(sw["threshold"], sw["f1"], lw=2, color=colors[i],
                    label=f"{r.dataset_name} (best {r.metrics['best_f1']:.3f} @ {r.metrics['best_f1_threshold']:.2f})")
        ax.axvline(self.score_threshold, color="black", ls="--", lw=1, label="operating threshold")
        ax.set_xlabel("Confidence threshold"); ax.set_ylabel("F1"); ax.set_ylim(0, 1.02)
        ax.set_title("F1 vs confidence threshold", fontweight="bold"); ax.legend(fontsize=8)
        done("comparison_f1_vs_threshold", fig)

        iou_results = [r for r in results if r.curves["ap_by_iou"]]
        if iou_results:
            fig, ax = plt.subplots(figsize=(11, 6))
            for i, r in enumerate(results):
                if r.curves["ap_by_iou"]:
                    ts, aps = zip(*r.curves["ap_by_iou"].items())
                    ax.plot(ts, aps, "o-", lw=2, color=colors[i], label=f"{r.dataset_name} (mAP={r.metrics['mAP50_95']:.3f})")
            ax.set_xlabel("IoU threshold"); ax.set_ylabel("AP"); ax.set_ylim(0, 1.02)
            ax.set_title("AP vs IoU threshold (localisation quality)", fontweight="bold"); ax.legend(fontsize=8)
            done("comparison_ap_vs_iou", fig)

        # ---- recall by size ---------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 6))
        buckets = [b for b, _, _ in SIZE_BUCKETS]
        width = 0.8 / len(results); x = np.arange(len(buckets))
        for i, r in enumerate(results):
            v = r.size_recall.set_index("bucket").reindex(buckets)["recall"].values.astype(float)
            ax.bar(x + i * width - 0.4 + width / 2, np.nan_to_num(v), width, color=colors[i], label=r.dataset_name)
        ax.set_xticks(x, buckets); ax.set_ylim(0, 1.05); ax.set_ylabel("Recall")
        ax.set_title("Recall by face size (empty bar = no faces in bucket)", fontweight="bold"); ax.legend(fontsize=8)
        done("comparison_recall_by_size", fig)

        # ---- speed --------------------------------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].bar(names, table["mean_inference_ms"], color=colors, yerr=None)
        axes[0].scatter(names, table["p95_inference_ms"], color="black", marker="_", s=400, label="p95")
        axes[0].set_ylabel("ms / image"); axes[0].set_title("Mean inference time (- = p95)"); axes[0].legend()
        axes[1].bar(names, table["images_per_second"], color=colors)
        axes[1].set_ylabel("images / second"); axes[1].set_title("Throughput")
        for ax in axes:
            ax.tick_params(axis="x", rotation=25)
        fig.suptitle("Inference speed comparison", fontweight="bold")
        done("comparison_inference_speed", fig)

        # ---- dataset statistics -----------------------------------------------
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        for ax, (col, title) in zip(axes, [("images", "Images evaluated"), ("faces", "GT faces"),
                                           ("mean_faces_per_image", "Mean faces / image"),
                                           ("median_face_size_px", "Median face size (px)")]):
            ax.bar(names, stats[col], color=colors); ax.set_title(title); ax.tick_params(axis="x", rotation=30)
        fig.suptitle("Dataset characteristics (after preprocessing)", fontweight="bold")
        done("comparison_dataset_statistics", fig)

        # ---- confusion matrix grid ---------------------------------------------
        fig, axes = plt.subplots(len(results), 2, figsize=(11, 4.2 * len(results)), squeeze=False)
        for i, r in enumerate(results):
            d, im = r.confusion["detection"], r.confusion["image_level"]
            self._draw_matrix(axes[i, 0], [[d["TP"], d["FN"]], [d["FP"], None]], ["Face", "Background"],
                              ["Face", "Background"], f"{r.dataset_name}: detection")
            self._draw_matrix(axes[i, 1], [[im["TP"], im["FN"]], [im["FP"], im["TN"]]], ["Has face", "No face"],
                              ["Face", "No face"], f"{r.dataset_name}: image-level")
        fig.suptitle(f"Confusion matrices @ score >= {self.score_threshold:.2f}", fontweight="bold", y=1.0)
        fig.tight_layout()
        done("comparison_confusion_matrices", fig)

        # ---- reports ----------------------------------------------------------
        self._write_markdown_report(results, table, stats, ranking, figs, out)
        pdf_path = self._write_pdf_report(results, table, figs, out)
        print(f"Comparison saved to {out}  |  PDF report: {pdf_path}")
        return table

    # --------------------------------------------------------
    @staticmethod
    def _md_table(df, float_fmt="{:.4f}"):
        def fmt(v):
            if isinstance(v, (float, np.floating)):
                return "n/a" if not np.isfinite(v) else float_fmt.format(v)
            return "n/a" if v is None else str(v)
        header = "| " + " | ".join(map(str, df.columns)) + " |"
        sep = "| " + " | ".join("---" for _ in df.columns) + " |"
        rows = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]
        return "\n".join([header, sep] + rows)

    def _write_markdown_report(self, results, table, stats, ranking, figs, out):
        cols = ["dataset", "images", "ground_truth_faces", "precision", "recall", "f1",
                "detection_accuracy", "AP_primary", "mAP50_95", "detection_roc_auc",
                "image_accuracy", "image_roc_auc", "count_accuracy", "mean_inference_ms"]
        lines = [
            "# RetinaFace evaluation report", "",
            f"- Operating score threshold: **{self.score_threshold}**  (curves use detections down to {self.inference_threshold})",
            f"- IoU threshold: **{self.iou_threshold}** (landmark containment for landmark-matched datasets)",
            "- Detection accuracy = TP / (TP + FP + FN); true negatives are undefined for detection.",
            "- Image accuracy = correct face / no-face decisions per image.", "",
            "## Ranking by F1", "", self._md_table(ranking), "",
            "## All metrics", "", self._md_table(table[[c for c in cols if c in table.columns]]), "",
            "## Dataset statistics", "", self._md_table(stats[[c for c in stats.columns if c not in ("description",)]][
                ["dataset", "images", "faces", "images_without_faces", "mean_faces_per_image",
                 "median_face_size_px", "match_mode"]]), "",
            "## Comparison figures", "",
        ]
        lines += [f"![{k}]({Path(p).name})" for k, p in figs.items()]
        lines += ["", "## Per-dataset figures", ""]
        for r in results:
            lines.append(f"### {r.dataset_name}")
            lines += [f"![{k}](../{r.dataset_name}/figures/{Path(p).name})" for k, p in r.figures.items()]
            lines.append("")
        (out / "evaluation_report.md").write_text("\n".join(lines), encoding="utf-8")

    def _write_pdf_report(self, results, table, figs, out):
        pdf_path = out / "evaluation_report.pdf"
        with PdfPages(pdf_path) as pdf:
            fig, ax = plt.subplots(figsize=(16, 2 + 0.5 * len(results))); ax.axis("off")
            cols = ["dataset", "precision", "recall", "f1", "detection_accuracy", "AP_primary",
                    "detection_roc_auc", "image_accuracy", "count_accuracy", "mean_inference_ms"]
            cell = [[(f"{v:.3f}" if isinstance(v, (float, np.floating)) and np.isfinite(v) else
                      ("n/a" if isinstance(v, (float, np.floating)) else str(v))) for v in row]
                    for row in table[cols].itertuples(index=False)]
            t = ax.table(cellText=cell, colLabels=cols, loc="center", cellLoc="center")
            t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.6)
            ax.set_title(f"RetinaFace evaluation summary (score >= {self.score_threshold}, IoU >= {self.iou_threshold})",
                         fontweight="bold", fontsize=14)
            pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)
            pages = list(figs.values())
            for r in results:
                pages += list(r.figures.values())
            for p in pages:
                img = plt.imread(p)
                fig, ax = plt.subplots(figsize=(img.shape[1] / 120, img.shape[0] / 120))
                ax.imshow(img); ax.axis("off")
                pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)
        return pdf_path


## Step 1: verify datasets on Kaggle

In [ ]:
# ============================================================
# STEP 1 - VERIFY EVERY DATASET ON KAGGLE
# Downloads each dataset and checks that its annotations parse.
# Only datasets marked VERIFIED are evaluated below.
# ============================================================
verification_df, dataset_roots = verify_kaggle_datasets(DATASETS)
display(verification_df[["dataset", "kaggle_handle", "downloaded", "annotations_parsed",
                         "annotated_images", "verified", "error"]])


## Step 2: evaluate each dataset

In [ ]:
# ============================================================
# STEP 2 - PREPROCESS -> FaceDataset -> RetinaFace.evaluate()
# Datasets are built and evaluated one at a time, then freed, so memory
# stays bounded. Swap `preprocess_face_sample` for your own function.
# ============================================================
detector = RetinaFace(
    score_threshold=SCORE_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    inference_threshold=INFERENCE_THRESHOLD,
    output_dir=OUTPUT_DIR,
)

results, run_errors = [], []
for cfg in DATASETS:
    if cfg["name"] not in dataset_roots:
        print(f"[SKIPPED] {cfg['name']}: failed Kaggle verification")
        continue
    try:
        dataset = build_dataset(cfg, dataset_roots[cfg["name"]],
                                preprocess_fn=preprocess_face_sample,
                                limit=MAX_IMAGES_PER_DATASET)
        print(dataset)
        results.append(detector.evaluate(dataset, save=True, plot=True, show=True))
    except Exception as exc:
        run_errors.append({"dataset": cfg["name"], "error": repr(exc)})
        print(f"[FAILED] {cfg['name']}: {exc}")
    finally:
        dataset = None
        gc.collect()

save_json(run_errors, OUTPUT_DIR / "evaluation_errors.json")


## Step 3: cross-dataset comparison

In [ ]:
# ============================================================
# STEP 3 - CROSS-DATASET COMPARISON
# Saves comparison CSV/JSON/Markdown, comparison figures and a combined
# PDF (every per-dataset and comparison figure) to data/comparison/.
# ============================================================
if results:
    comparison_df = detector.compare(results, show=True)
    display(comparison_df[["dataset", "images", "ground_truth_faces", "precision", "recall", "f1",
                           "detection_accuracy", "AP_primary", "mAP50_95", "detection_roc_auc",
                           "image_accuracy", "count_accuracy", "mean_inference_ms"]].round(4))
else:
    print("No dataset was evaluated - check data/dataset_verification.csv")


## Saved files

## Output layout

```
data/
├── dataset_verification.csv          # Kaggle verification result per dataset
├── evaluation_errors.json
├── <dataset>/
│   ├── metrics.json / metrics.csv     # every scalar metric + confusion matrices + dataset stats
│   ├── per_image_metrics.csv          # TP/FP/FN, counts, IoU, timing per image
│   ├── detections.csv                 # every detection with score, box, IoU, TP flag
│   ├── pr_curve.csv  froc_curve.csv  roc_curve_detection.csv  roc_curve_image.csv
│   ├── threshold_sweep.csv  ap_by_iou.csv  recall_by_face_size.csv  confusion_matrices.json
│   └── figures/                       # 11 PNG figures (summary, confusion, ROC, PR, sweep, FROC, ...)
└── comparison/
    ├── metrics_comparison.csv/.json  ranking_by_f1.csv  dataset_statistics.csv
    ├── comparison_*.png               # bar, heatmap, radar, PR/FROC/ROC overlays, size, speed, CMs
    ├── evaluation_report.md
    └── evaluation_report.pdf          # all comparison + per-dataset figures in one file
```

In [ ]:
# ============================================================
# SAVED OUTPUT FILES
# ============================================================
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path):<75} {path.stat().st_size:>12,} bytes")


## Evaluate your own dataset

In [ ]:
# ============================================================
# EVALUATE YOUR OWN DATASET
# ============================================================
# 1) (optional) define your own preprocessing with the same signature:
#
# def my_preprocess(image, boxes, landmarks=None):
#     image, boxes, landmarks = preprocess_face_sample(image, boxes, landmarks, max_side=1024)
#     return image, boxes, landmarks
#
# 2) build the FaceDataset outside the model, then evaluate it:
#
# custom_images = sorted(Path("/kaggle/input/my-dataset/images").glob("*.jpg"))
# custom_annotations = {
#     "image1.jpg": [[10, 20, 150, 190]],     # pixel xyxy
#     "image2.jpg": [],                       # image without faces (enables image-level ROC)
# }
# my_dataset = build_custom_dataset("my_dataset", custom_images, custom_annotations,
#                                   preprocess_fn=my_preprocess)
# my_result = detector.evaluate(my_dataset)
# detector.compare(results + [my_result], output_name="comparison_with_custom")
#
# Already have arrays in memory? Construct the container directly:
# my_dataset = FaceDataset("my_dataset", images=list_of_rgb_arrays,
#                          boxes=list_of_Nx4_arrays, image_ids=list_of_names)
